In [3]:
from pathlib import Path
import json, pandas as pd, numpy as np

DATA = Path('./data/')


schema = json.loads((DATA / 'feature_schema_context.json').read_text(encoding='utf-8'))
df = pd.read_parquet(DATA / 'model_features_context.parquet')
print(df.shape)                            # (2629440, 56)

# pd.set_option("display.max_columns", None)
# df
# 
INPUTS = schema['inputs']          # 입력 39개 (직접 고르지 말고 이 목록을 쓰기)
TARGET = 'rentals_target_1h'       # 반납 모델은 'returns_target_1h'

use = df.coverage_status.eq('observed_operating') & df[TARGET].notna()
train = df[use & df.split.eq('train')]
valid = df[use & df.split.eq('validation')]
test  = df[use & df.split.eq('test')]      # 마지막까지 열어보지 않기
print(len(train), len(valid), len(test))

(2629440, 56)
1569378 526178 524875


In [4]:
import json
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna

warnings.filterwarnings('ignore', message=".*'eval_set' is deprecated.*")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# =====================================================================
# 설정
# =====================================================================
RESULTS = Path('./results/')
MODEL_NAME = 'LightGBM_Optuna_v2'   # 기존 결과를 덮어쓰지 않도록 이름 변경
SEED = 42

# 최종 모델 pkl 저장 위치
MODEL_DIR = Path('./models/')
MODEL_DIR.mkdir(exist_ok=True)

N_TRIALS = 40          # 30 → 40
TIMEOUT = 40 * 60      # 20분 → 40분
SAMPLE_FRAC = 0.1      # 0.3 → 0.1

INPUTS = schema['inputs']
TARGETS = ['rentals_target_1h', 'returns_target_1h']
CAT_COLS = ['station_group', 'region', 'commute_period', 'holiday_type']

# =====================================================================
# 데이터 분할
# =====================================================================
use = df.coverage_status.eq('observed_operating') & df['rentals_target_1h'].notna() & df['returns_target_1h'].notna()
train_mask = use & df.split.eq('train')
valid_mask = use & df.split.eq('validation')
test_mask  = use & df.split.eq('test')
print(f"Train {train_mask.sum():,} / Validation {valid_mask.sum():,} / Test {test_mask.sum():,}")

# 범주형 변환을 한 번에 해서 train/valid/test 의 카테고리 목록을 맞춤
X_all = df.loc[use, INPUTS].copy()
for col in CAT_COLS:
    X_all[col] = X_all[col].astype('category')
X_train = X_all.loc[train_mask[use]]
X_valid = X_all.loc[valid_mask[use]]
X_test  = X_all.loc[test_mask[use]]
del X_all

# 탐색용 train 샘플 (두 타깃 공통)
search_idx = X_train.sample(frac=SAMPLE_FRAC, random_state=SEED).index

def score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)  # 음수 예측은 0으로 clip
    err = y_pred - y_true
    total = np.abs(y_true).sum()
    return {
        'MAE': np.abs(err).mean(),
        'RMSE': np.sqrt((err ** 2).mean()),
        'WAPE': np.abs(err).sum() / total if total else np.nan,
        'Bias': err.mean(),
    }

FIXED_PARAMS = dict(objective='regression', n_estimators=1500, subsample_freq=1,
                    random_state=SEED, verbosity=-1)

def suggest_params(trial):
    return dict(
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        num_leaves=trial.suggest_int('num_leaves', 15, 255, log=True),
        min_child_samples=trial.suggest_int('min_child_samples', 10, 300, log=True),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_alpha=trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    )

def fit_lgb(params, X_tr, y_tr, X_va, y_va):
    model = lgb.LGBMRegressor(**FIXED_PARAMS, **params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)])
    return model

segments = {
    '전체': (X_valid.index, X_test.index),
    '출근': (X_valid.index[df.loc[X_valid.index, 'commute_period'] == 'morning'],
             X_test.index[df.loc[X_test.index, 'commute_period'] == 'morning']),
    '퇴근': (X_valid.index[df.loc[X_valid.index, 'commute_period'] == 'evening'],
             X_test.index[df.loc[X_test.index, 'commute_period'] == 'evening']),
    '역세권': (X_valid.index[df.loc[X_valid.index, 'station_group'] == '역세권'],
               X_test.index[df.loc[X_test.index, 'station_group'] == '역세권']),
    '업무지구': (X_valid.index[df.loc[X_valid.index, 'station_group'] == '업무지구'],
                 X_test.index[df.loc[X_test.index, 'station_group'] == '업무지구']),
    '주거지': (X_valid.index[df.loc[X_valid.index, 'station_group'] == '주거지'],
               X_test.index[df.loc[X_test.index, 'station_group'] == '주거지']),
}

metrics_records, pred_df_list = [], []
best_params_by_target, search_info = {}, {}

# =====================================================================
# 타깃별: Optuna 탐색 → 최적 파라미터로 train 전체 재학습 → pkl 저장 → 평가
# =====================================================================
for target in TARGETS:
    y_train = df.loc[X_train.index, target]
    y_valid = df.loc[X_valid.index, target]

    # 1) Optuna 탐색 — validation MAE 최소화 (test 는 사용하지 않음)
    print(f"\n🔍 Optuna search for {target} (max {N_TRIALS} trials / {TIMEOUT // 60} min)...")
    X_search, y_search = X_train.loc[search_idx], y_train.loc[search_idx]

    def objective(trial):
        model = fit_lgb(suggest_params(trial), X_search, y_search, X_valid, y_valid)
        trial.set_user_attr('best_iteration', int(model.best_iteration_))
        return score(y_valid, model.predict(X_valid))['MAE']

    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=SEED),
                                study_name=f'{MODEL_NAME}_{target}')
    t0 = time.time()
    study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT)
    search_seconds = time.time() - t0
    print(f"   ↳ {len(study.trials)} trials, {search_seconds:.0f}s, best validation MAE (sample) = {study.best_value:.4f}")
    print(f"   ↳ best params: {study.best_params}")

    best_params_by_target[target] = study.best_params
    search_info[target] = {'n_trials_done': len(study.trials), 'search_seconds': round(search_seconds, 1),
                           'best_value_validation_MAE_on_sample': study.best_value}

    # 2) 최적 파라미터로 train 전체 재학습
    print(f"🚀 Training final LightGBM for {target}...")
    start_time = time.time()
    model = fit_lgb(study.best_params, X_train, y_train, X_valid, y_valid)
    train_seconds = time.time() - start_time
    print(f"⏱️ Final training completed in {train_seconds:.2f} seconds (best_iteration={model.best_iteration_}).")
    best_params_by_target[target] = {**study.best_params, 'best_iteration': int(model.best_iteration_)}

    # 최종 모델을 바로 pkl 로 저장 (재학습 없이 평가에 쓰는 모델 그대로)
    model_path = MODEL_DIR / f'bicycle_{MODEL_NAME}_{target}.pkl'
    joblib.dump(model, model_path)
    print(f"💾 saved {model_path} (best_iteration={model.best_iteration_})")

    # 3) 예측 및 세그먼트 평가
    valid_pred_series = pd.Series(np.clip(model.predict(X_valid), 0, None), index=X_valid.index)
    test_pred_series  = pd.Series(np.clip(model.predict(X_test), 0, None), index=X_test.index)

    for seg_name, (v_idx, t_idx) in segments.items():
        for split_name, idx, pred in [('validation', v_idx, valid_pred_series), ('test', t_idx, test_pred_series)]:
            m = score(df.loc[idx, target], pred.loc[idx])
            metrics_records.append({
                'model': MODEL_NAME, 'target': target, 'split': split_name, 'segment': seg_name,
                'n': len(idx), 'MAE': m['MAE'], 'RMSE': m['RMSE'], 'WAPE': m['WAPE'], 'Bias': m['Bias'],
                'train_seconds': train_seconds,
            })

    for split_name, pred in [('validation', valid_pred_series), ('test', test_pred_series)]:
        sub_pred = df.loc[pred.index, ['station_id', 'timestamp']].copy()
        sub_pred['split'] = split_name
        sub_pred['target'] = target
        sub_pred['y_true'] = df.loc[pred.index, target].astype(float)
        sub_pred['y_pred'] = pred.values.astype(float)
        pred_df_list.append(sub_pred)

# =====================================================================
# 결과 저장 (results/)
# =====================================================================
print("\n💾 Writing outputs to results/ ...")
pd.DataFrame(metrics_records).to_csv(RESULTS / f'metrics_{MODEL_NAME}.csv', index=False)
pd.concat(pred_df_list, ignore_index=True).to_parquet(RESULTS / f'pred_{MODEL_NAME}.parquet', index=False, engine='pyarrow')

params_dict = {
    "inputs": INPUTS,
    "hyperparameters": {
        "fixed": {k: v for k, v in FIXED_PARAMS.items() if k != 'verbosity'},
        "early_stopping_rounds": 30,
        "best_by_target": best_params_by_target,
        "optuna": {"sampler": "TPESampler", "seed": SEED, "metric": "validation MAE",
                   "n_trials": N_TRIALS, "timeout_seconds": TIMEOUT, "search_sample_frac": SAMPLE_FRAC,
                   "by_target": search_info},
    },
}
with open(RESULTS / f'params_{MODEL_NAME}.json', 'w', encoding='utf-8') as f:
    json.dump(params_dict, f, indent=2, ensure_ascii=False)

print("🎯 Finished: Optuna-tuned LightGBM pipeline complete.")


Train 1,569,378 / Validation 526,178 / Test 524,875

🔍 Optuna search for rentals_target_1h (max 40 trials / 40 min)...
   ↳ 16 trials, 2564s, best validation MAE (sample) = 1.4159
   ↳ best params: {'learning_rate': 0.01709602693868314, 'num_leaves': 205, 'min_child_samples': 28, 'subsample': 0.9695303487517939, 'colsample_bytree': 0.7543323327269813, 'reg_alpha': 0.0313346372357801, 'reg_lambda': 1.3174505301240464}
🚀 Training final LightGBM for rentals_target_1h...
⏱️ Final training completed in 238.03 seconds (best_iteration=1500).
💾 saved models\bicycle_LightGBM_Optuna_v2_rentals_target_1h.pkl (best_iteration=1500)

🔍 Optuna search for returns_target_1h (max 40 trials / 40 min)...
   ↳ 15 trials, 2460s, best validation MAE (sample) = 1.4252
   ↳ best params: {'learning_rate': 0.014913447638126187, 'num_leaves': 157, 'min_child_samples': 11, 'subsample': 0.947845524961155, 'colsample_bytree': 0.5289951188075321, 'reg_alpha': 0.1977369220562478, 'reg_lambda': 0.27892094739170353}
🚀 T

In [5]:
import pandas as pd

# 1. Load the generated metrics file from your DATA path
metrics_path = RESULTS / f'metrics_{MODEL_NAME}.csv'
metrics_df = pd.read_csv(metrics_path)

# 2. Separate Validation and Test for clean viewing
val_summary = metrics_df[metrics_df['split'] == 'validation'].copy()
test_summary = metrics_df[metrics_df['split'] == 'test'].copy()

# 3. Display Validation Dashboard (Use this to compare against Team A's Baseline)
print("=================== 📊 VALIDATION PERFORMANCE ===================")
val_pivot = val_summary.pivot(index='segment', columns='target', values=['MAE', 'WAPE', 'Bias'])
display(val_pivot.round(4))

# 4. Display Final Test Dashboard (The 2025 Evaluation - Checked Once)
print("\n=================== 🏁 FINAL 2025 TEST PERFORMANCE ===================")
test_pivot = test_summary.pivot(index='segment', columns='target', values=['MAE', 'WAPE', 'Bias'])
display(test_pivot.round(4))


=================== 📊 VALIDATION PERFORMANCE ===================


MAE                                WAPE  \
target  rentals_target_1h returns_target_1h rentals_target_1h   
segment                                                         
업무지구               1.2787            1.3203            0.4429   
역세권                1.6613            1.6716            0.3818   
전체                 1.3659            1.3847            0.4313   
주거지                1.1576            1.1620            0.5118   
출근                 1.6136            1.6752            0.3804   
퇴근                 2.0651            2.0581            0.3412   

                                       Bias                    
target  returns_target_1h rentals_target_1h returns_target_1h  
segment                                                        
업무지구               0.4353            0.0405            0.0009  
역세권                0.3749            0.1055            0.0357  
전체                 0.4246            0.0509            0.0159  
주거지                0.5072            0.0066            0.0112  
출근                 0.3365           -0.0723           -0.0822  
퇴근                 0.3427            0.1581            0.0415


=================== 🏁 FINAL 2025 TEST PERFORMANCE ===================


MAE                                WAPE  \
target  rentals_target_1h returns_target_1h rentals_target_1h   
segment                                                         
업무지구               1.1699            1.1930            0.4886   
역세권                1.4955            1.5091            0.4267   
전체                 1.2418            1.2549            0.4772   
주거지                1.0604            1.0631            0.5554   
출근                 1.4715            1.5404            0.4047   
퇴근                 1.8925            1.8540            0.3873   

                                       Bias                    
target  returns_target_1h rentals_target_1h returns_target_1h  
segment                                                        
업무지구               0.4719            0.1891            0.1207  
역세권                0.4168            0.2529            0.1422  
전체                 0.4669            0.1802            0.1197  
주거지                0.5546            0.0987            0.0963  
출근                 0.3534            0.0757            0.0399  
퇴근                 0.3918            0.4501            0.3130